In [1]:
!pip install reportlab

In [23]:
import os
import time
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from reportlab.platypus import SimpleDocTemplate, Paragraph, PageBreak
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.lib.pagesizes import A4

In [10]:
caegories = []
official_link = "https://macommunaute.ca/bottin-des-organismes/?mc_regions=cote-des-neiges-notre-dame-de-grace"

Links =[
    "https://macommunaute.ca/bottin-des-organismes/?mc_regions=cote-des-neiges-notre-dame-de-grace&mc_thematiques=vie-quartier",
    "https://macommunaute.ca/bottin-des-organismes/?mc_regions=cote-des-neiges-notre-dame-de-grace&mc_thematiques=sports-loisirs",
    "https://macommunaute.ca/bottin-des-organismes/?mc_regions=cote-des-neiges-notre-dame-de-grace&mc_thematiques=sante",
    "https://macommunaute.ca/bottin-des-organismes/?mc_regions=cote-des-neiges-notre-dame-de-grace&mc_thematiques=retraite",
    "https://macommunaute.ca/bottin-des-organismes/?mc_regions=cote-des-neiges-notre-dame-de-grace&mc_thematiques=logement",
    "https://macommunaute.ca/bottin-des-organismes/?mc_regions=cote-des-neiges-notre-dame-de-grace&mc_thematiques=justice-droits",
    "https://macommunaute.ca/bottin-des-organismes/?mc_regions=cote-des-neiges-notre-dame-de-grace&mc_thematiques=famille",
    "https://macommunaute.ca/bottin-des-organismes/?mc_regions=cote-des-neiges-notre-dame-de-grace&mc_thematiques=emploi"

]

#site1 = "https://macommunaute.ca/bottin-des-organismes/?mc_regions=cote-des-neiges-notre-dame-de-grace&mc_thematiques=alimentation"
#site1 = "https://macommunaute.ca/bottin-des-organismes/?mc_regions=cote-des-neiges-notre-dame-de-grace&mc_thematiques=sports-loisirs"
#site1 = "https://macommunaute.ca/bottin-des-organismes/?mc_regions=cote-des-neiges-notre-dame-de-grace&mc_thematiques=emploi"
site1 = "https://macommunaute.ca/bottin-des-organismes/?mc_regions=cote-des-neiges-notre-dame-de-grace&mc_thematiques=sante"

In [11]:
def get_total_results(driver):
    # Extraire le nombre total de résultats
    try:
        resultats_div = driver.find_element(By.CSS_SELECTOR, ".resultats")
        resultats_text = resultats_div.text
        print(f"Total de résultats: {resultats_text}")
    except Exception as e:
        print(f"Erreur lors de la récupération du nombre de résultats: {e}")

In [12]:
# option1
# only works when one pages returned
def get_cards(driver):
    """Extract all cards returned by the filter search"""
    try:
        cards = WebDriverWait(driver, 15).until(
            EC.presence_of_all_elements_located((By.CSS_SELECTOR, ".card-post.card-post-fullwidth"))
)
        print(f"Nombre de cartes trouvées: {len(cards)}")
        return cards
    except Exception as e:
        print(f"Erreur lors de la récupération des cartes: {e}")
        return []
    
def get_cards_url(driver, cards):
    # Collect all hrefs from card elements
    return [card.get_attribute("href") for card in cards if card.get_attribute("href")]

    
#option2
# works with pagination
def get_all_cards_urls(driver):
    all_urls = []

    while True:
        # Attendre les cartes
        WebDriverWait(driver, 15).until(
            EC.presence_of_all_elements_located(
                (By.CSS_SELECTOR, ".card-post.card-post-fullwidth")
            )
        )

        cards = driver.find_elements(By.CSS_SELECTOR, ".card-post.card-post-fullwidth")
        print(f"Cartes trouvées sur cette page: {len(cards)}")

        # Extraire les href directement
        for card in cards:
            href = card.get_attribute("href")
            if href and href not in all_urls:
                all_urls.append(href)

        # Pagination
        try:
            next_link = driver.find_element(By.CSS_SELECTOR, "a.next.page-numbers")
            next_url = next_link.get_attribute("href")
            print(" Navigation vers:", next_url)
            driver.get(next_url)

        except:
            print(" Plus de page suivante")
            break

    print(f"TOTAL URLs récupérées: {len(all_urls)}")
    return all_urls



In [20]:
def scrape_card_details(driver, url):
    """Open a card URL in the current tab, scrape details, and return data without printing."""
    wait = WebDriverWait(driver, 10)
    driver.get(url)
    time.sleep(2)  # Wait for page to load
    try:
        section = wait.until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "div.section-apropos.active")))       
        full_text = section.text.strip()
        
        title = driver.find_element(By.CSS_SELECTOR, "h1.small.coul-blanc").text.strip()
        website_link = None
        links = driver.find_elements(By.CSS_SELECTOR, "a.bold")
        for link in links:
            if "consulter le site web" in link.text.lower():
                website_link = link.get_attribute("href")
                break

        return {
            "Organisation": title,
            "url": url,
            "details": full_text,
            "website_link": website_link
        }
    except Exception as e:
        print(f"Erreur lors de la récupération des détails pour {url}: {e}")
        return {
            "url": url,
            "details": None
        }

In [14]:
chrome_opts = Options()
#chrome_opts.add_argument("--headless=new")  # supprimez cette ligne si tu veux voir le navigateur
chrome_opts.add_argument("--no-sandbox")
chrome_opts.add_argument("--disable-dev-shm-usage")
driver = webdriver.Chrome(options=chrome_opts)

In [16]:
driver.get(site1)

In [17]:
total_results = get_total_results(driver)


Total de résultats: 65 Organismes à but non lucratif


In [9]:
#cards = get_cards(driver)
#cards_url = get_cards_url(driver, cards)
#cards_url
#card_0 = scrape_card_details(driver, cards_url[0])
#card_0
#card_0["details"]

In [18]:
cards_urls = get_all_cards_urls(driver)

Cartes trouvées sur cette page: 16
 Navigation vers: https://macommunaute.ca/bottin-des-organismes/page/2/?mc_regions=cote-des-neiges-notre-dame-de-grace&mc_thematiques=sante
Cartes trouvées sur cette page: 16
 Navigation vers: https://macommunaute.ca/bottin-des-organismes/page/3/?mc_regions=cote-des-neiges-notre-dame-de-grace&mc_thematiques=sante
Cartes trouvées sur cette page: 16
 Navigation vers: https://macommunaute.ca/bottin-des-organismes/page/4/?mc_regions=cote-des-neiges-notre-dame-de-grace&mc_thematiques=sante
Cartes trouvées sur cette page: 16
 Navigation vers: https://macommunaute.ca/bottin-des-organismes/page/5/?mc_regions=cote-des-neiges-notre-dame-de-grace&mc_thematiques=sante
Cartes trouvées sur cette page: 1
 Plus de page suivante
TOTAL URLs récupérées: 65


In [11]:
cards_urls

['https://macommunaute.ca/bottin-des-organismes/centrederepitphilou/',
 'https://macommunaute.ca/bottin-des-organismes/associationdespinabifidaetdhydrocephalieduquebec/',
 'https://macommunaute.ca/bottin-des-organismes/centredebenevolatsarpad/',
 'https://macommunaute.ca/bottin-des-organismes/a-deux-mains/',
 'https://macommunaute.ca/bottin-des-organismes/actioncancerduseindemontreal/',
 'https://macommunaute.ca/bottin-des-organismes/amiquebecagircontrelamaladiementale/',
 'https://macommunaute.ca/bottin-des-organismes/associationpolioquebec/',
 'https://macommunaute.ca/bottin-des-organismes/atlas-centre-de-pediatrie-sociale-en-communaute-de-cote-des-neiges/',
 'https://macommunaute.ca/bottin-des-organismes/bureaudesdrthomassindryaremkoomnipraticiens/',
 'https://macommunaute.ca/bottin-des-organismes/cansupport-des-cedres/',
 'https://macommunaute.ca/bottin-des-organismes/centrecommunautairejuridiquedemontrealaidejuridique/',
 'https://macommunaute.ca/bottin-des-organismes/centredactio

In [21]:
results = []

for card in cards_urls:
    details = scrape_card_details(driver, card)
    print(details)
    if details:
        results.append(details)


{'Organisation': 'Centre Philou', 'url': 'https://macommunaute.ca/bottin-des-organismes/centrederepitphilou/', 'details': "À PROPOS DE L'ORGANISME\nCoordonnées\n2450, ch. de la Côte-Sainte-Catherine,\nMontréal QC H3T 1B1\n514-739-4861\nEnvoyer un message\nConsulter le site Web\nHeures d'ouverture\nHORAIRE\nAdministration: Lundi au vendredi 9h à 17h\nGarderie: 9h à 17h\nÉcole: 9h à 16h\nRépit: 24/7\nContacts (1)\nCommuniquer avec ce contact\nSignaler une erreur dans cette fiche\nMission\nLe Centre Philou a pour mission d'offrir des services d'aide et de développement en continu aux enfants polyhandicapés et leur famille, en complémentarité avec les services publics.\nDescription\nLe Centre Philou est un organisme sans but lucratif et une œuvre de bienfaisance qui offre des services spécialisés aux enfants polyhandicapés et leur famille : séjours de répit, programmes de développement, garderie et soutien familial.\nPartager la fiche:\nFacebook\nLinkedIn\nTwitter\nMessenger\nPrint\nShare"

In [22]:
with open("macommunaute_sante.txt", "w", encoding="utf-8") as f:
    for i, card in enumerate(results, 1):
        f.write(f"===== Organisation {i} =====\n")
        f.write(f"Page URL: {card['url']}\n")
        f.write(f"Website: {card['website_link']}\n\n")
        f.write(card["details"])   #
        f.write("\n" + "="*50 + "\n\n\n")  # Separator between organisations


In [24]:
def save_to_pdf(results, filename="results.pdf"):
    doc = SimpleDocTemplate(filename, pagesize=A4)
    styles = getSampleStyleSheet()
    story = []

    for i, card in enumerate(results, 1):
        details = card["details"].replace("\n", "<br/>")

        text = (
            f"<b>Organisation {i}</b><br/>"
            f"<b>Page URL:</b> {card['url']}<br/>"
            f"<b>Website:</b> {card['website_link']}<br/><br/>"
            f"{details}"
        )

        story.append(Paragraph(text, styles["Normal"]))
        story.append(PageBreak())
        
    print("PDF will be saved in:", os.getcwd())
    doc.build(story)
save_to_pdf(results, "macommunaute_sante.pdf")


PDF will be saved in: d:\mscudemproject\main\dynamic_rags\scrap_data
